In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Pair‑wise AUC comparison with a correct implementation of DeLong’s test
"""

import os
import numpy as np
import pandas as pd
import scipy.stats
from scipy.stats import norm          # NEW
from sklearn.metrics import roc_auc_score
from IPython.display import display, HTML

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
OUTPUT_CSV_FILE = 'delong_comparison_results.csv'

MODELS_TO_EXCLUDE = [
    'preop_ASA Rule',
    'combined_ASA Rule',
    'combined_Hybrid (MLP + LSTM)'
]

model_translate = {
    'log_reg': 'Logistic Regression', 'xgb': 'GBT',
    'svm_linear': 'SVM (Linear)', 'mlp': 'MLP', 'rf': 'Random Forest',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AutoGluon',
    'lstm': 'LSTM', 'hybrid': 'Hybrid (MLP + LSTM)'
}

model_base_map = {'LSTM': 'base_54k', 'Hybrid (MLP + LSTM)': 'base_54k'}
DEFAULT_BASE = 'base'

# =============================================================================
# FAST & MEMORY‑SAFE DELONG (no big matrices)
# =============================================================================
from scipy.stats import norm

def _structural_components(y_true: np.ndarray,
                           scores:   np.ndarray):
    """
    Return (auc, v10, v01) without building an n_pos × n_neg table.

    v10[i]  = mean_j θ( s_pos[i] , s_neg[j] )
    v01[j]  = mean_i θ( s_pos[i] , s_neg[j] ),     θ = 1, 0.5, 0
    """
    y_true  = y_true.astype(int).ravel()
    scores  = scores.ravel()
    pos     = scores[y_true == 1]
    neg     = scores[y_true == 0]

    n_pos   = pos.size
    n_neg   = neg.size
    if n_pos == 0 or n_neg == 0:
        raise ValueError("Need at least one positive and one negative sample.")

    # Pre‑sort once for binary‑search look‑ups
    neg_sorted = np.sort(neg)
    pos_sorted = np.sort(pos)

    # --- V10: for every positive score, compare with all negatives -----------
    left  = np.searchsorted(neg_sorted, pos, side='left')    # #neg < s_pos
    right = np.searchsorted(neg_sorted, pos, side='right')   # #neg ≤ s_pos
    ties  = right - left
    v10   = (left + 0.5 * ties) / n_neg          # shape (n_pos,)

    # --- V01: for every negative score, compare with all positives -----------
    left  = np.searchsorted(pos_sorted, neg, side='left')    # #pos < s_neg
    right = np.searchsorted(pos_sorted, neg, side='right')   # #pos ≤ s_neg
    ties  = right - left
    greater = n_pos - right
    v01   = (greater + 0.5 * ties) / n_pos       # shape (n_neg,)

    auc = v10.mean()                             # identical to roc_auc_score
    return auc, v10, v01


def delong_test(y_true:  np.ndarray,
                scores1: np.ndarray,
                scores2: np.ndarray):
    """
    DeLong’s test for correlated AUCs (two‑sided).
    Returns (auc1, auc2, p_value).
    Memory complexity: O(n) ;  never builds n_pos × n_neg arrays.
    """
    auc1, v10_1, v01_1 = _structural_components(y_true, scores1)
    auc2, v10_2, v01_2 = _structural_components(y_true, scores2)

    n_pos = v10_1.size
    n_neg = v01_1.size

    s10 = np.cov(np.vstack([v10_1, v10_2]), ddof=1)
    s01 = np.cov(np.vstack([v01_1, v01_2]), ddof=1)

    var_auc_diff = (s10[0, 0] + s10[1, 1] - 2 * s10[0, 1]) / n_pos + \
                   (s01[0, 0] + s01[1, 1] - 2 * s01[0, 1]) / n_neg

    if var_auc_diff <= 0 or not np.isfinite(var_auc_diff):
        return auc1, auc2, 1.0          # numerical safeguard

    z = (auc1 - auc2) / np.sqrt(var_auc_diff)
    p = 2 * (1 - norm.cdf(abs(z)))
    return auc1, auc2, p

# =============================================================================
# DATA PREPARATION AND ANALYSIS FUNCTIONS
# =============================================================================
def prepare_and_flatten_data(data_sources):
    """Loads and flattens predictions for each requested source."""
    prepared_data = {}
    print("--- Preparing and Loading Data ---")
    for source_name in data_sources:
        file_path = os.path.join(RESULTS_DIR, f"tabular_{source_name}_test.pkl")
        try:
            df = pd.read_pickle(file_path)
            print(f"Loaded: {file_path}")
        except FileNotFoundError:
            print(f"ERROR: {file_path} not found → skipped.")
            continue

        # Ground‑truth arrays (“base” models) ---------------------------
        base_models_data = {}
        base_df = df[df['model_name'].str.contains('base', na=False)]
        for _, row in base_df.iterrows():
            base_models_data[row['model_name']] = np.concatenate(row['y_pred_binary'])

        # Predictive models --------------------------------------------
        models_df = df[~df['model_name'].str.contains('base', na=False)]
        for _, row in models_df.iterrows():
            model_key = row['model_name']
            if model_key not in model_translate:
                continue

            model_name_trans = model_translate[model_key]
            full_name = f"{source_name}_{model_name_trans}"

            base_to_use = model_base_map.get(model_name_trans, DEFAULT_BASE)
            if base_to_use not in base_models_data:
                print(f"WARNING: base '{base_to_use}' missing for {full_name} → skipped.")
                continue

            y_true_flat = base_models_data[base_to_use]
            y_prob_flat = np.concatenate(row['y_prob'])

            if len(y_true_flat) != len(y_prob_flat):
                print(f"WARNING: length mismatch for {full_name} → skipped.")
                continue

            prepared_data[full_name] = (y_true_flat, y_prob_flat)

    print(f"Prepared data for {len(prepared_data)} models.")
    return prepared_data

def run_comprehensive_delong_tests(prepared_data):
    """Runs DeLong’s test for every model pair that shares identical ground truth."""
    print("--- Running Pairwise DeLong Tests ---")
    all_models = sorted(m for m in prepared_data if m not in MODELS_TO_EXCLUDE)

    if not all_models:
        print("ERROR: No models left to compare.")
        return None

    grid = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
    tot = len(all_models) ** 2
    done = 0

    for m1 in all_models:
        for m2 in all_models:
            done += 1
            print(f"[{done}/{tot}] {m1} vs {m2}")
            if m1 == m2:
                grid.loc[m1, m2] = np.nan
                continue

            y1, p1 = prepared_data[m1]
            y2, p2 = prepared_data[m2]

            if not np.array_equal(y1, y2):
                print("   → GT mismatch, set N/A.")
                grid.loc[m1, m2] = "N/A"
                continue

            _, _, p = delong_test(y1, p1, p2)
            grid.loc[m1, m2] = p

    return grid

def format_and_save_results(p_value_grid):
    """Pretty‑print, write CSV, and (in Jupyter) show an HTML table."""
    print("--- Formatting Results ---")

    def fmt(p):
        if pd.isna(p): return "nan"
        if isinstance(p, str): return p
        if p < 1e-4: return "<0.0001***"
        if p < 1e-3: return f"{p:.4f}***"
        if p < 1e-2: return f"{p:.4f}**"
        if p < 5e-2: return f"{p:.4f}*"
        return f"{p:.4f}"

    text_grid = p_value_grid.map(fmt)
    print("\nDeLong Test Results:\n")
    print(text_grid.to_string())

    text_grid.to_csv(OUTPUT_CSV_FILE)
    print(f"\nCSV written to {OUTPUT_CSV_FILE}")

    # Jupyter‑friendly styled table
    def highlight(p):
        return 'background-color: lightgreen' if isinstance(p, (float, int)) and p < 0.05 else ''

    styled = (
        p_value_grid.style
        .map(highlight)
        .format(fmt)
        .set_caption("Pairwise DeLong Test P‑Values")
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]},
            {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '5px')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'),
                                             ('font-size', '1.2em'),
                                             ('font-weight', 'bold')]}
        ])
    )
    try:
        display(styled)
    except NameError:
        # display() unavailable outside Jupyter – ignore.
        pass

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    data_to_compare = prepare_and_flatten_data(['preop', 'combined'])

    if data_to_compare:
        p_grid = run_comprehensive_delong_tests(data_to_compare)
        if p_grid is not None:
            format_and_save_results(p_grid)
    else:
        print("No data prepared – exiting.")


--- Preparing and Loading Data ---
Loaded: /home/server/Projects/data/AKI/results/tabular_preop_test.pkl
Loaded: /home/server/Projects/data/AKI/results/tabular_combined_test.pkl
Prepared data for 15 models.
--- Running Pairwise DeLong Tests ---
[1/144] combined_AutoGluon vs combined_AutoGluon
[2/144] combined_AutoGluon vs combined_GBT
[3/144] combined_AutoGluon vs combined_KNN
[4/144] combined_AutoGluon vs combined_Logistic Regression
[5/144] combined_AutoGluon vs combined_MLP
[6/144] combined_AutoGluon vs combined_Random Forest
[7/144] combined_AutoGluon vs preop_AutoGluon
[8/144] combined_AutoGluon vs preop_GBT
[9/144] combined_AutoGluon vs preop_KNN
[10/144] combined_AutoGluon vs preop_Logistic Regression
[11/144] combined_AutoGluon vs preop_MLP
[12/144] combined_AutoGluon vs preop_Random Forest
[13/144] combined_GBT vs combined_AutoGluon
[14/144] combined_GBT vs combined_GBT
[15/144] combined_GBT vs combined_KNN
[16/144] combined_GBT vs combined_Logistic Regression
[17/144] combine

,combined_AutoGluon,combined_GBT,combined_KNN,combined_Logistic Regression,combined_MLP,combined_Random Forest,preop_AutoGluon,preop_GBT,preop_KNN,preop_Logistic Regression,preop_MLP,preop_Random Forest
combined_AutoGluon,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.0834,<0.0001***,0.5669,<0.0001***,<0.0001***,<0.0001***,0.3820
combined_GBT,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
combined_KNN,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
combined_Logistic Regression,<0.0001***,<0.0001***,<0.0001***,nan,0.4340,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.2469,<0.0001***
combined_MLP,<0.0001***,<0.0001***,<0.0001***,0.4340,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.6381,<0.0001***
combined_Random Forest,0.0834,<0.0001***,<0.0001***,<0.0001***,<0.0001***,nan,<0.0001***,0.3963,<0.0001***,<0.0001***,<0.0001***,0.3863
preop_AutoGluon,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
preop_GBT,0.5669,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.3963,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,0.7383
preop_KNN,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,nan,0.0005***,<0.0001***,<0.0001***
preop_Logistic Regression,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.0005***,nan,<0.0001***,<0.0001***


In [7]:
import pandas as pd
import numpy as np
import scipy.stats
from sklearn.metrics import roc_auc_score
import os
from IPython.display import display, HTML

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
# --- Paths and Naming ---
# Note: Adjust this path if the script is not in your project's root directory.
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
OUTPUT_CSV_FILE = 'delong_comparison_results.csv'

# --- Models to Exclude from the Final Grid ---
MODELS_TO_EXCLUDE = [
    'preop_ASA Rule',
    'combined_ASA Rule',
    'combined_Hybrid (MLP + LSTM)'
]

# --- Model Name Translations for Plots ---
model_translate = {
    'log_reg': 'Logistic Regression', 'xgb': 'GBT',
    'svm_linear': 'SVM (Linear)', 'mlp': 'MLP', 'rf': 'Random Forest',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AutoGluon',
    'lstm': 'LSTM', 'hybrid': 'Hybrid (MLP + LSTM)'
    # Note: 'base' and 'base_54k' are handled separately as ground truth sources.
}

# --- Ground-truth 'base' model mapping ---
model_base_map = {
    'LSTM': 'base_54k',
    'Hybrid (MLP + LSTM)': 'base_54k'
}
DEFAULT_BASE = 'base'

# =============================================================================
# MEMORY-EFFICIENT DELONG'S TEST IMPLEMENTATION
# =============================================================================
def fast_delong_test(y_true, y_scores_1, y_scores_2):
    """
    Computes the p-value from DeLong's test for two correlated ROC curves.
    This version is memory-efficient and avoids creating large intermediate matrices.
    """
    y_true = np.asarray(y_true)
    y_scores_1 = np.asarray(y_scores_1)
    y_scores_2 = np.asarray(y_scores_2)

    # Sort scores and labels based on ground truth
    order = np.argsort(y_true)
    y_true = y_true[order]
    y_scores_1 = y_scores_1[order]
    y_scores_2 = y_scores_2[order]

    n1 = (y_true == 1).sum() # Number of positive samples
    n2 = (y_true == 0).sum() # Number of negative samples
    
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan, np.nan

    # Get ranks of scores
    rank_1 = scipy.stats.rankdata(y_scores_1)
    rank_2 = scipy.stats.rankdata(y_scores_2)

    # Calculate AUROCs using Mann-Whitney U-statistic from ranks
    auc1 = (np.sum(rank_1[y_true == 1]) - n1 * (n1 + 1) / 2) / (n1 * n2)
    auc2 = (np.sum(rank_2[y_true == 1]) - n1 * (n1 + 1) / 2) / (n1 * n2)

    # Calculate structural components V10 and V01
    v10_1 = (rank_1[y_true == 1] - np.mean(rank_1[y_true == 1])) / (n1 - 1)
    v01_1 = (rank_1[y_true == 0] - np.mean(rank_1[y_true == 0])) / (n2 - 1)
    v10_2 = (rank_2[y_true == 1] - np.mean(rank_2[y_true == 1])) / (n1 - 1)
    v01_2 = (rank_2[y_true == 0] - np.mean(rank_2[y_true == 0])) / (n2 - 1)
    
    # Calculate covariance matrix S
    s10 = np.cov(v10_1, v10_2)
    s01 = np.cov(v01_1, v01_2)
    
    # Variance of the AUC difference
    var_diff = s10[0, 0] / n1 + s01[0, 0] / n2 - 2 * s10[0, 1] / n1 - 2 * s01[0, 1] / n2 \
             + s10[1, 1] / n1 + s01[1, 1] / n2

    if var_diff <= 0:
        return auc1, auc2, 1.0

    z_score = (auc1 - auc2) / np.sqrt(var_diff)
    p_value = 2.0 * (1 - scipy.stats.norm.cdf(abs(z_score)))

    return auc1, auc2, p_value


# =============================================================================
# DATA PREPARATION AND ANALYSIS FUNCTIONS
# =============================================================================

def prepare_and_flatten_data(data_sources):
    """
    Loads model results, flattens predictions from all folds into a single
    array, and returns a dictionary mapping model names to their data.
    """
    prepared_data = {}
    print("--- Preparing and Loading Data ---")
    for source_name in data_sources:
        file_path = os.path.join(RESULTS_DIR, f"tabular_{source_name}_test.pkl")
        try:
            df = pd.read_pickle(file_path)
            print(f"Successfully loaded: {file_path}")
        except FileNotFoundError:
            print(f"ERROR: Results file not found at {file_path}. Skipping.")
            continue
            
        # Store all ground truth 'base' models first
        base_models_data = {}
        base_df = df[df['model_name'].str.contains('base', na=False)]
        for _, row in base_df.iterrows():
             # y_pred_binary contains the true labels
            base_models_data[row['model_name']] = np.concatenate(row['y_pred_binary'])

        # Process the predictive models
        models_df = df[~df['model_name'].str.contains('base', na=False)]
        for _, row in models_df.iterrows():
            model_key = row['model_name']
            if model_key not in model_translate:
                continue
                
            model_name_translated = model_translate[model_key]
            full_model_name = f"{source_name}_{model_name_translated}"
            
            # Determine which base model's ground truth to use
            base_to_use = model_base_map.get(model_name_translated, DEFAULT_BASE)
            if base_to_use not in base_models_data:
                print(f"WARNING: Base model '{base_to_use}' for model '{full_model_name}' not found. Skipping.")
                continue

            y_true_flat = base_models_data[base_to_use]
            y_prob_flat = np.concatenate(row['y_prob'])
            
            # Ensure consistent lengths
            if len(y_true_flat) != len(y_prob_flat):
                print(f"WARNING: Mismatched sample count for '{full_model_name}'. Skipping.")
                continue

            prepared_data[full_model_name] = (y_true_flat, y_prob_flat)
            
    print(f"Prepared data for {len(prepared_data)} models.")
    print("--- Data Preparation Complete ---\n")
    return prepared_data


def run_comprehensive_delong_tests(prepared_data):
    """
    Runs DeLong's test for all pairs of models (preop vs. preop, 
    combined vs. combined, and preop vs. combined).
    """
    print("--- Running Comprehensive Pairwise DeLong's Tests ---")
    all_models_raw = sorted(prepared_data.keys())
    
    # Filter out models specified in the exclusion list
    all_models = [model for model in all_models_raw if model not in MODELS_TO_EXCLUDE]
    print(f"Filtered out {len(all_models_raw) - len(all_models)} models. Running comparisons on {len(all_models)} models.")
    
    if not all_models:
        print("ERROR: No models left to compare after filtering.")
        return None

    # Initialize DataFrame to store p-values
    results_grid = pd.DataFrame(index=all_models, columns=all_models, dtype=float)

    total_comparisons = len(all_models) * len(all_models)
    current_comparison = 0

    for model_1_name in all_models:
        for model_2_name in all_models:
            current_comparison += 1
            print(f"Running comparison {current_comparison}/{total_comparisons}: {model_1_name} vs. {model_2_name}")
            
            if model_1_name == model_2_name:
                results_grid.loc[model_1_name, model_2_name] = np.nan
                continue

            y_true_1, y_prob_1 = prepared_data[model_1_name]
            y_true_2, y_prob_2 = prepared_data[model_2_name]

            # Critical validation: DeLong's test requires comparison on the same set of labels
            if not np.array_equal(y_true_1, y_true_2):
                print(f"--> WARNING: Ground truth mismatch between {model_1_name} and {model_2_name}. Cannot compare.")
                results_grid.loc[model_1_name, model_2_name] = "N/A"
                continue
            
            # Use the first ground truth array since they are identical
            _, _, p_value = fast_delong_test(y_true_1, y_prob_1, y_prob_2)
            results_grid.loc[model_1_name, model_2_name] = p_value

    print("\n--- Pairwise Testing Complete ---\n")
    return results_grid

def format_and_save_results(p_value_grid):
    """
    Formats the p-value grid for console, CSV, and rich HTML display in Jupyter.
    """
    print("--- Formatting and Saving Results ---")
    
    # --- 1. For CSV and Console (Text with stars) ---
    def format_p_value_text(p):
        if pd.isna(p): return "nan"
        if isinstance(p, str): return p
        if p < 0.0001: return "<0.0001***"
        if p < 0.001: return f"{p:.4f}***"
        if p < 0.01: return f"{p:.4f}**"
        if p < 0.05: return f"{p:.4f}*"
        return f"{p:.4f}"

    # MODIFIED: Changed .applymap() to .map()
    formatted_text_grid = p_value_grid.map(format_p_value_text)
    print("DeLong Test Results (All Models):")
    print(formatted_text_grid.to_string())
    try:
        formatted_text_grid.to_csv(OUTPUT_CSV_FILE)
        print(f"\nSuccessfully saved text results to: {OUTPUT_CSV_FILE}")
    except Exception as e:
        print(f"\nError saving results to CSV: {e}")

    # --- 2. For Jupyter Notebook (Styled HTML Table) ---
    def style_significant_p(p):
        """Highlights cells with a significant p-value."""
        if isinstance(p, (int, float)) and p < 0.05:
            return 'background-color: lightgreen'
        return ''

    # Create the styled table
    # We use the raw p-values for styling and the formatted text for display
    # MODIFIED: Changed .applymap() to .map() for the Styler object
    styled_grid = p_value_grid.style \
        .map(style_significant_p) \
        .format(formatter=format_p_value_text) \
        .set_caption("Pairwise DeLong Test P-Values") \
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]},
            {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '5px')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '1.2em'), ('font-weight', 'bold')]}
        ])
    
    print("\nDisplaying styled HTML table for Jupyter Notebook:")
    display(styled_grid)


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    # 1. Load data for the sources we want to compare
    data_to_compare = prepare_and_flatten_data(['preop', 'combined'])
    
    if data_to_compare:
        # 2. Run the pairwise comparisons
        p_value_results = run_comprehensive_delong_tests(data_to_compare)
        
        if p_value_results is not None:
            # 3. Format, save, and display the final grid
            format_and_save_results(p_value_results)
    else:
        print("\nAnalysis could not proceed due to lack of prepared data.")


--- Preparing and Loading Data ---
Successfully loaded: /home/server/Projects/data/AKI/results/tabular_preop_test.pkl
Successfully loaded: /home/server/Projects/data/AKI/results/tabular_combined_test.pkl
Prepared data for 15 models.
--- Data Preparation Complete ---

--- Running Comprehensive Pairwise DeLong's Tests ---
Filtered out 3 models. Running comparisons on 12 models.
Running comparison 1/144: combined_AutoGluon vs. combined_AutoGluon
Running comparison 2/144: combined_AutoGluon vs. combined_GBT
Running comparison 3/144: combined_AutoGluon vs. combined_KNN
Running comparison 4/144: combined_AutoGluon vs. combined_Logistic Regression
Running comparison 5/144: combined_AutoGluon vs. combined_MLP
Running comparison 6/144: combined_AutoGluon vs. combined_Random Forest
Running comparison 7/144: combined_AutoGluon vs. preop_AutoGluon
Running comparison 8/144: combined_AutoGluon vs. preop_GBT
Running comparison 9/144: combined_AutoGluon vs. preop_KNN
Running comparison 10/144: combine

,combined_AutoGluon,combined_GBT,combined_KNN,combined_Logistic Regression,combined_MLP,combined_Random Forest,preop_AutoGluon,preop_GBT,preop_KNN,preop_Logistic Regression,preop_MLP,preop_Random Forest
combined_AutoGluon,nan,0.7914,0.0369*,0.5854,0.6492,0.9358,0.6582,0.9788,0.4322,0.4433,0.6743,0.9677
combined_GBT,0.7914,nan,0.0255*,0.4432,0.4909,0.7330,0.8387,0.7774,0.3379,0.3505,0.5250,0.7822
combined_KNN,0.0369*,0.0255*,nan,0.0688,0.0773,0.0346*,0.0248*,0.0510,0.1801,0.1491,0.1019,0.0534
combined_Logistic Regression,0.5854,0.4432,0.0688,nan,0.9711,0.6266,0.3524,0.6319,0.6894,0.6218,0.9573,0.6188
combined_MLP,0.6492,0.4909,0.0773,0.9711,nan,0.6988,0.4180,0.6702,0.6764,0.7177,0.9826,0.6806
combined_Random Forest,0.9358,0.7330,0.0346*,0.6266,0.6988,nan,0.5566,0.9687,0.4701,0.4836,0.7164,0.9679
preop_AutoGluon,0.6582,0.8387,0.0248*,0.3524,0.4180,0.5566,nan,0.5080,0.1976,0.2029,0.3461,0.3159
preop_GBT,0.9788,0.7774,0.0510,0.6319,0.6702,0.9687,0.5080,nan,0.3908,0.4163,0.6323,0.9877
preop_KNN,0.4322,0.3379,0.1801,0.6894,0.6764,0.4701,0.1976,0.3908,nan,0.8710,0.6161,0.4150
preop_Logistic Regression,0.4433,0.3505,0.1491,0.6218,0.7177,0.4836,0.2029,0.4163,0.8710,nan,0.6286,0.4077


# Testing DeLong Function

In [3]:
import numpy as np
from mlstatkit.toolkit import Delong_test
from sklearn.metrics import roc_auc_score

def check_delong_test():
    """
    A simple script to verify that the Delong_test function is working.
    """
    print("--- DeLong Test Function Checker ---")

    # 1. Create some simple, reproducible data
    # Ground truth labels
    target = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 1])

    # Predictions from a "good" model (Model A)
    preds_a = np.array([0.1, 0.2, 0.9, 0.8, 0.3, 0.7, 0.2, 0.85, 0.95, 0.6])

    # Predictions from a "slightly worse" model (Model B)
    preds_b = np.array([0.2, 0.3, 0.8, 0.7, 0.4, 0.6, 0.3, 0.75, 0.85, 0.5])
    
    # Predictions from a "random" model (Model C)
    preds_c = np.array([0.5, 0.4, 0.6, 0.3, 0.7, 0.2, 0.8, 0.1, 0.9, 0.0])


    print(f"\nModel A AUROC: {roc_auc_score(target, preds_a):.4f}")
    print(f"Model B AUROC: {roc_auc_score(target, preds_b):.4f}")
    print(f"Model C AUROC: {roc_auc_score(target, preds_c):.4f}")
    
    # 2. Run the DeLong test
    try:
        # --- Test 1: Compare two similar models ---
        print("\nComparing Model A vs. Model B (similar performance)...")
        # We expect the p-value to be high (e.g., > 0.05), indicating no significant difference
        # MODIFIED: Changed to positional arguments (target, preds1, preds2)
        statistic_ab, p_value_ab = Delong_test(target, preds_a, preds_b)
        print(f"  > DeLong Test Result: p-value = {p_value_ab:.4f}")
        if p_value_ab > 0.05:
            print("  > As expected, the difference is NOT statistically significant.")
        else:
            print("  > The difference is statistically significant (which is unexpected for this test case).")

        # --- Test 2: Compare two different models ---
        print("\nComparing Model A vs. Model C (different performance)...")
        # We expect the p-value to be lower, indicating a potentially significant difference
        # MODIFIED: Changed to positional arguments (target, preds1, preds2)
        statistic_ac, p_value_ac = Delong_test(target, preds_a, preds_c)
        print(f"  > DeLong Test Result: p-value = {p_value_ac:.4f}")
        if p_value_ac < 0.05:
            print("  > As expected, the difference IS statistically significant.")
        else:
            print("  > The difference is NOT statistically significant (which is plausible depending on the data).")


        print("\nConclusion: The Delong_test function executed successfully.")

    except ImportError:
        print("\nERROR: Could not import Delong_test. Please ensure 'mlstatkit' is installed (`pip install mlstatkit`).")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")

if __name__ == "__main__":
    check_delong_test()


--- DeLong Test Function Checker ---

Model A AUROC: 1.0000
Model B AUROC: 1.0000
Model C AUROC: 0.2500

Comparing Model A vs. Model B (similar performance)...
  > DeLong Test Result: p-value = nan
  > The difference is statistically significant (which is unexpected for this test case).

Comparing Model A vs. Model C (different performance)...
  > DeLong Test Result: p-value = 0.0000
  > As expected, the difference IS statistically significant.

Conclusion: The Delong_test function executed successfully.


/home/server/Projects/VitalDB-Dimensionality-Reduction/data_postprocessing/mlstatkit/toolkit.py:85: RuntimeWarning: invalid value encountered in divide
  z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, delongcov), l.T)).flatten()
